In [12]:
# Step 1: Import required dependencies.
import sys
from pathlib import Path

import torch

# Step 1.1: Add repository root to sys.path for notebook execution.
repo_root = Path.cwd().resolve()
while not (repo_root / "pyproject.toml").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from fermigates.calibration.linear_calibration import LinearCalibration
from fermigates.experiments import Experiment
from fermigates.gates import FermiGate
from fermigates.losses import fermiloss
from fermigates.models import MLP


In [13]:
# Step 2: Build deterministic vanilla MLP model.
torch.manual_seed(7)
vanilla_model = MLP(
    input_dim=784,
    hidden_dims=[128, 64],
    output_dim=10,
    gate=None,
    loss=fermiloss,
    calibration=LinearCalibration(d_in=10, d_out=10, learnable=False),
)


In [26]:
# Step 3: Build deterministic weight-gated MLP model.
torch.manual_seed(7)
gated_model = MLP(
    input_dim=784,
    hidden_dims=[128, 64],
    output_dim=10,
    gate=lambda: FermiGate(
        mode="neuron",
        annealer="linear",
        init_mu=0.0,
        init_temperature=0.0,
    ),
    loss=fermiloss,
    calibration=LinearCalibration(d_in=10, d_out=10, learnable=False),
)


In [27]:
# Step 4: Create experiments for vanilla and gated models.
vanilla_exp = Experiment(
    model=vanilla_model,
    dataset="mnist",
    epochs=3,
    learning_rate=1e-3,
)
gated_exp = Experiment(
    model=gated_model,
    dataset="mnist",
    epochs=3,
    learning_rate=1e-3,
)


In [28]:
# Step 5: Train both models with explicit calibration policy.
print("\n===== Training Vanilla MLP =====")
vanilla_exp.train()
print("\n===== Training Weight-Gated MLP =====")
gated_exp.train()



===== Training Vanilla MLP =====

===== Training Weight-Gated MLP =====


In [ ]:
# Step 6: Evaluate and print accuracy and sparsity metrics.
vanilla_acc = vanilla_exp.accuracy()
gated_acc = gated_exp.accuracy()

print("\n===== Evaluation =====")
print(f"Vanilla Accuracy: {vanilla_acc:.4f}")
print(f"Gated Accuracy:   {gated_acc:.4f}")


===== Evaluation =====
Vanilla Accuracy: 0.9778
Gated Accuracy:   0.9980


In [18]:
# Step 3: Build deterministic weight-gated MLP model.
torch.manual_seed(7)
gated_model = MLP(
    input_dim=784,
    hidden_dims=[128, 64],
    output_dim=10,
    gate=lambda: FermiGate(
        mode="weight",
        annealer="linear",
        init_mu=0.5,
        init_temperature=1.0,
    ),
    loss=fermiloss,
    calibration=LinearCalibration(d_in=10, d_out=10, learnable=False),
)

gated_exp = Experiment(
    model=gated_model,
    dataset="mnist",
    epochs=3,
    learning_rate=1e-3,
)

gated_exp.train(calibrate_after_first_epoch=True, ridge_lambda=1e-3)
print("\n===== Sparsity Report =====")
gated_weight_sparsity = gated_exp.weight_sparsity_metrics(threshold=0.5)
print(f"Gated   Weight-Gate Sparsity: {gated_weight_sparsity}")


===== Sparsity Report =====
Gated   Weight-Gate Sparsity: {'kept': 399, 'total': 109184, 'fraction_kept': 0.003654381594372802, 'fraction_sparse': 0.9963456184056272}
